In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema_source", "bronze")
dbutils.widgets.text("esquema_sink", "silver")

In [0]:
catalogo = dbutils.widgets.get("catalogo")
esquema_source = dbutils.widgets.get("esquema_source")
esquema_sink = dbutils.widgets.get("esquema_sink")

In [0]:
df_clientes = spark.table(f"{catalogo}.{esquema_source}.clientes")
df_polizas = spark.table(f"{catalogo}.{esquema_source}.polizas")

In [0]:

silver_polizas_enriq = (
    df_polizas.alias("p")
    .join(df_clientes.alias("c"), F.col("p.id_cliente") == F.col("c.id_cliente"), "left")
    .select(
        F.col("p.id_poliza"),
        F.col("p.id_cliente"),
        F.initcap(F.trim(F.col("c.nombre_completo"))).alias("nombre_completo_final"),
        F.lower(F.trim(F.col("c.correo"))).alias("correo_final"),
        F.coalesce(F.nullif(F.trim(F.col("c.ciudad")), F.lit("")), F.lit("DESCONOCIDO")).alias("ciudad_final"),
        F.to_date("c.fecha_nacimiento").alias("fecha_nacimiento"),
        F.to_date("c.fecha_registro").alias("fecha_registro"),
        F.to_date("p.fecha_inicio").alias("fecha_inicio"),
        F.col("p.tipo_seguro"),
        F.col("p.estado").alias("estado_poliza"),
        F.col("p.prima_anual").cast("double").alias("prima_anual"),
        F.floor(F.datediff(F.current_date(), F.to_date("c.fecha_nacimiento")) / F.lit(365.25)).alias("edad_cliente"),
        F.datediff(F.current_date(), F.to_date("c.fecha_registro")).alias("antiguedad_dias_cliente"),
    )
)


In [0]:

silver_polizas_enriq.write.mode("overwrite").insertInto(f"{catalogo}.{esquema_sink}.clientes_polizas_transformed")